In [1]:
# CELL 1 — Import Libraries
# This cell imports core libraries needed for data manipulation, file management, and model loading.
import pandas as pd
import numpy as np
import joblib
import os
from google.colab import files

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# CELL 2 — Upload the Trained PKL Model
# This cell uses Google Colab's file upload tool to upload the trained model pipeline.
print("Please upload: alumni_donor_model_pipeline.pkl")
uploaded = files.upload()

# Automatically identify the uploaded filename
model_file = next(iter(uploaded))
print(f"Uploaded filename: {model_file}")

Please upload: alumni_donor_model_pipeline.pkl


Saving alumni_donor_model_pipeline (2).pkl to alumni_donor_model_pipeline (2).pkl
Uploaded filename: alumni_donor_model_pipeline (2).pkl


In [3]:
# CELL 3 — Load the Trained Model
# This cell loads the uploaded pipeline model into memory using joblib.
model = joblib.load(model_file)

print("Model loaded successfully!")
print("Model type:", type(model))

Model loaded successfully!
Model type: <class 'sklearn.pipeline.Pipeline'>


In [4]:
# CELL 4 — Verify the Model
# This cell checks that the loaded object is ready to make predictions.
has_predict = hasattr(model, "predict")
has_predict_proba = hasattr(model, "predict_proba")

if has_predict and has_predict_proba:
    print("✅ Trained model is fully verified and ready for prediction!")
else:
    print("❌ The loaded object does not support model predictions.")

✅ Trained model is fully verified and ready for prediction!


In [5]:
# CELL 5 — Create a Sample Alumni Input
# This cell defines a single profile representing a test alumnus as raw inputs.
sample_alumni = {
    "graduation_year": 2015,
    "age": 32,
    "events_attended": 5,
    "emails_received": 100,
    "emails_opened": 70,
    "newsletter_clicks": 8,
    "volunteer_events": 2,
    "previous_donations": 3,
    "total_donation_amount": 15000,
    "donation_frequency": 2,
    "days_since_last_donation": 200,
    "career_level": "Senior",
    "industry": "IT",
    "days_since_last_interaction": 30
}

input_data = pd.DataFrame([sample_alumni])
display(input_data)

,graduation_year,age,events_attended,emails_received,emails_opened,newsletter_clicks,volunteer_events,previous_donations,total_donation_amount,donation_frequency,days_since_last_donation,career_level,industry,days_since_last_interaction
0,2015,32,5,100,70,8,2,3,15000,2,200,Senior,IT,30


In [6]:
# CELL 6 — Perform the Same Feature Engineering
# This cell transforms raw inputs using the exact same formulas used during training.
input_data["years_since_graduation"] = 2026 - input_data["graduation_year"]

input_data["email_open_rate"] = np.where(
    input_data["emails_received"] > 0,
    input_data["emails_opened"] / input_data["emails_received"],
    0
)
input_data["email_open_rate"] = input_data["email_open_rate"].clip(0, 1)

input_data["average_donation_amount"] = np.where(
    input_data["previous_donations"] > 0,
    input_data["total_donation_amount"] / input_data["previous_donations"],
    0
)

input_data["donation_recency_score"] = 1 / (1 + input_data["days_since_last_donation"])

input_data["event_engagement"] = input_data["events_attended"] / 10
input_data["newsletter_engagement"] = input_data["newsletter_clicks"] / 15
input_data["volunteer_engagement"] = input_data["volunteer_events"] / 5

input_data["interaction_recency"] = (1 - input_data["days_since_last_interaction"] / 500).clip(0, 1)

input_data["engagement_score"] = (
    0.30 * input_data["event_engagement"]
    + 0.30 * input_data["email_open_rate"]
    + 0.15 * input_data["newsletter_engagement"]
    + 0.15 * input_data["volunteer_engagement"]
    + 0.10 * input_data["interaction_recency"]
) * 100

display(input_data)

,graduation_year,age,events_attended,emails_received,emails_opened,newsletter_clicks,volunteer_events,previous_donations,total_donation_amount,donation_frequency,...,days_since_last_interaction,years_since_graduation,email_open_rate,average_donation_amount,donation_recency_score,event_engagement,newsletter_engagement,volunteer_engagement,interaction_recency,engagement_score
0,2015,32,5,100,70,8,2,3,15000,2,...,30,11,0.7,5000.0,0.004975,0.5,0.533333,0.4,0.94,59.4


In [7]:
# CELL 7 — Make Prediction
# This cell predicts the class of the single input sample.
prediction = model.predict(input_data)[0]

print("Predicted Class:", prediction)
print("Interpretation:")
print("0 = Did not donate")
print("1 = Donated")

Predicted Class: 1
Interpretation:
0 = Did not donate
1 = Donated


In [8]:
# CELL 8 — Calculate Donation Probability
# This cell calculates and outputs the donation propensity percentage score.
probability = model.predict_proba(input_data)[0][1]
propensity_score = probability * 100

print(f"Propensity Score: {propensity_score:.2f}%")
print("\n⚠️ Note: This propensity score is an estimated likelihood, not a guarantee.")

Propensity Score: 95.18%

⚠️ Note: This propensity score is an estimated likelihood, not a guarantee.


In [9]:
# CELL 9 — Create Propensity Category
# This cell groups the output score into high, medium, or low categories.
if propensity_score >= 80:
    category = "High"
    interpretation = "Likely to Donate"
elif propensity_score >= 50:
    category = "Medium"
    interpretation = "Moderate Donation Likelihood"
else:
    category = "Low"
    interpretation = "Lower Donation Likelihood"

print("Propensity Score:", round(propensity_score, 2), "%")
print("Category:", category)
print("Interpretation:", interpretation)

Propensity Score: 95.18 %
Category: High
Interpretation: Likely to Donate


In [10]:
# CELL 10 — Test Multiple Alumni Examples
# This cell runs 3 distinct alumni scenarios through transformation and prediction.
test_cases = [
    {
        "name": "Highly Engaged Alumni",
        "graduation_year": 2020,
        "age": 30,
        "events_attended": 8,
        "emails_received": 40,
        "emails_opened": 35,
        "newsletter_clicks": 12,
        "volunteer_events": 5,
        "previous_donations": 5,
        "total_donation_amount": 5000,
        "donation_frequency": 4,
        "days_since_last_donation": 30,
        "career_level": "Executive",
        "industry": "Technology",
        "days_since_last_interaction": 10
    },
    {
        "name": "Moderately Engaged Alumni",
        "graduation_year": 2015,
        "age": 35,
        "events_attended": 4,
        "emails_received": 25,
        "emails_opened": 12,
        "newsletter_clicks": 5,
        "volunteer_events": 2,
        "previous_donations": 2,
        "total_donation_amount": 1000,
        "donation_frequency": 1,
        "days_since_last_donation": 200,
        "career_level": "Senior",
        "industry": "Finance",
        "days_since_last_interaction": 100
    },
    {
        "name": "Low Engagement Alumni",
        "graduation_year": 2005,
        "age": 45,
        "events_attended": 0,
        "emails_received": 20,
        "emails_opened": 2,
        "newsletter_clicks": 0,
        "volunteer_events": 0,
        "previous_donations": 0,
        "total_donation_amount": 0,
        "donation_frequency": 0,
        "days_since_last_donation": 1000,
        "career_level": "Mid",
        "industry": "Manufacturing",
        "days_since_last_interaction": 450
    }
]

results = []
for case in test_cases:
    case_df = pd.DataFrame([case]).drop(columns=["name"])

    # Feature Engineering
    case_df["years_since_graduation"] = 2026 - case_df["graduation_year"]
    case_df["email_open_rate"] = np.where(case_df["emails_received"] > 0, case_df["emails_opened"] / case_df["emails_received"], 0).clip(0, 1)
    case_df["average_donation_amount"] = np.where(case_df["previous_donations"] > 0, case_df["total_donation_amount"] / case_df["previous_donations"], 0)
    case_df["donation_recency_score"] = 1 / (1 + case_df["days_since_last_donation"])
    case_df["event_engagement"] = case_df["events_attended"] / 10
    case_df["newsletter_engagement"] = case_df["newsletter_clicks"] / 15
    case_df["volunteer_engagement"] = case_df["volunteer_events"] / 5
    case_df["interaction_recency"] = (1 - case_df["days_since_last_interaction"] / 500).clip(0, 1)
    case_df["engagement_score"] = (
        0.30 * case_df["event_engagement"]
        + 0.30 * case_df["email_open_rate"]
        + 0.15 * case_df["newsletter_engagement"]
        + 0.15 * case_df["volunteer_engagement"]
        + 0.10 * case_df["interaction_recency"]
    ) * 100

    # Prediction
    prob = model.predict_proba(case_df)[0][1] * 100

    if prob >= 80:
        cat = "High"
        interp = "Likely to Donate"
    elif prob >= 50:
        cat = "Medium"
        interp = "Moderate Donation Likelihood"
    else:
        cat = "Low"
        interp = "Lower Donation Likelihood"

    results.append({
        "Alumni": case["name"],
        "Propensity Score": f"{prob:.2f}%",
        "Category": cat,
        "Interpretation": interp
    })

display(pd.DataFrame(results))

,Alumni,Propensity Score,Category,Interpretation
0,Highly Engaged Alumni,98.03%,High,Likely to Donate
1,Moderately Engaged Alumni,85.10%,High,Likely to Donate
2,Low Engagement Alumni,49.35%,Low,Lower Donation Likelihood


In [11]:
# CELL 11 — Test the Saved Model Again
# This cell re-loads the PKL model file to confirm loading integrity.
reloaded_model = joblib.load(model_file)
reloaded_prediction = reloaded_model.predict(input_data)[0]
print("Saved model prediction successful!")

Saved model prediction successful!


In [13]:
# CELL 12 — Final Model Test Summary
# This cell provides a concise structural summary of the pipeline's purpose and layout.
print("--- FINAL MODEL TEST SUMMARY ---")
print("Model name: Logistic Regression")
print("Task: Binary Classification")
print("Prediction target: donated_next_12_months")
print("Output: Donation probability / propensity score")
print("Categories: High, Medium, Low")
print("Model source:", model_file)
print("\nThis notebook only tests the trained model. Model training was completed in Notebook 2.")

--- FINAL MODEL TEST SUMMARY ---
Model name: Logistic Regression
Task: Binary Classification
Prediction target: donated_next_12_months
Output: Donation probability / propensity score
Categories: High, Medium, Low
Model source: alumni_donor_model_pipeline (2).pkl

This notebook only tests the trained model. Model training was completed in Notebook 2.


In [14]:
from google.colab import files

uploaded = files.upload()

Saving alumni_donor_model_pipeline (3).pkl to alumni_donor_model_pipeline (3).pkl


In [15]:
import joblib

model = joblib.load("alumni_donor_model_pipeline.pkl")

FileNotFoundError: [Errno 2] No such file or directory: 'alumni_donor_model_pipeline.pkl'